In [9]:
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.metrics import classification_report
from preprocess import preprocess


# =========================
# 1. Load dataset
# =========================

df = pd.read_json("../donnees/articles.json")
# df = pd.read_csv("../donnees/Le360.com.csv")
# category_mapping = {
#     "Business": "اقتصاد",
#     "Culture": "ثقافة",
#     "International": "دولي",
#     "Sport": "رياضة",
#     "Policy": "سياسة",
#     "Society": "مجتمع",
# }
# df = df[df["Category"].isin(category_mapping)]
# df["Category"] = df["Category"].map(category_mapping)

X = df["body"]
y = df["categories"].str[0]


# =========================
# 2. Keep final test set
# =========================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


# =========================
# 3. Preprocessing
# =========================

X_train = X_train.apply(preprocess)
X_test = X_test.apply(preprocess)


# =========================
# 4. 5-fold cross-validation
# =========================

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

fold_scores = []


for fold, (train_index, val_index) in enumerate(
    skf.split(X_train, y_train),
    start=1
):

    X_fold_train = X_train.iloc[train_index]
    X_fold_val = X_train.iloc[val_index]

    y_fold_train = y_train.iloc[train_index]
    y_fold_val = y_train.iloc[val_index]


    # =========================
    # 5. TF-IDF for this fold
    # =========================

    vectorizer = TfidfVectorizer(
        min_df=2
    )

    X_fold_train_tfidf = vectorizer.fit_transform(
        X_fold_train
    )

    X_fold_val_tfidf = vectorizer.transform(
        X_fold_val
    )


    # =========================
    # 6. Logistic Regression
    # =========================

    lr_model = LogisticRegression(
        C=1.0,
        max_iter=10000
    )

    lr_model.fit(
        X_fold_train_tfidf,
        y_fold_train
    )


    # =========================
    # 7. Validation prediction
    # =========================

    y_fold_pred = lr_model.predict(
        X_fold_val_tfidf
    )
    errors = pd.DataFrame({
    "text": X_fold_val.reset_index(drop=True),
    "actual": y_fold_val.reset_index(drop=True),
    "predicted": y_fold_pred
    })

    errors = errors[errors["actual"] != errors["predicted"]]

    print("Number of errors:", len(errors))

    print("\nPolitique predicted as Society:")
    print(
        errors[
            (errors["actual"] == "سياسة") &
            (errors["predicted"] == "مجتمع")
        ][["actual", "predicted", "text"]].head(5)
    )

    print("\nSociety predicted as Politique:")
    print(
        errors[
            (errors["actual"] == "مجتمع") &
            (errors["predicted"] == "سياسة")
        ][["actual", "predicted", "text"]].head(5)
    )


    print(
        classification_report(
            y_fold_val,
            y_fold_pred,
            digits=4
        )
    )




    # =========================
    # 8. Macro F1
    # =========================

    fold_f1 = f1_score(
        y_fold_val,
        y_fold_pred,
        average="macro"
    )

    fold_scores.append(fold_f1)

    print(
        f"Fold {fold}: Macro F1 = {fold_f1:.4f}"
    )


# =========================
# 9. CV result
# =========================

print("\nCross-validation results:")

print(
    f"Mean Macro F1: "
    f"{sum(fold_scores) / len(fold_scores):.4f}"
)

print(
    f"Standard deviation: "
    f"{pd.Series(fold_scores).std():.4f}"
)

Number of errors: 377

Politique predicted as Society:
    actual predicted                                               text
30   سياسة     مجتمع  دعت وزارة داخلية مختلف اجهزة امنية مغربية والس...
79   سياسة     مجتمع  صادق مجلس حكومة منعقد يوم اربعاء، مشروع مرسوم ...
324  سياسة     مجتمع  عثر حرس مدني، امس بثغر سبتة محتل، مخبا للاسلحة...
598  سياسة     مجتمع  كشف حبوب شرقاوي، مدير مكتب مركزي للابحاث قضائي...
658  سياسة     مجتمع  اشرف مدير عام للمديرية عامة للامن وطني، عبد لط...

Society predicted as Politique:
    actual predicted                                               text
90   مجتمع     سياسة  بعد ايام قليلة فقط ضجة تي اثارها صحافي مصري، ذ...
339  مجتمع     سياسة  اعلنت نقابة وطنية لاطباء عيون بالقطاع خاص، است...
414  مجتمع     سياسة  افادت مصادر عليمة لLe ، ان مواطنين فرنسيين معت...
486  مجتمع     سياسة  وجهه حاضر باستمرار شاشة واسمه كل لسان. محمد يو...
617  مجتمع     سياسة  تشبع شاب بفكر وتقنيات حرب عصابات، اذ تخصص مناه...
              precision    recall  f1-score   su

In [10]:
print("Development shape:", X_train.shape)
print("Development labels:", y_train.shape)
print("Number of samples:", len(X_train))
print("\nCategory distribution:")
print(y_train.value_counts())

Development shape: (16800,)
Development labels: (16800,)
Number of samples: 16800

Category distribution:
categories
ثقافة     2800
دولي      2800
اقتصاد    2800
رياضة     2800
سياسة     2800
مجتمع     2800
Name: count, dtype: int64


In [11]:
for fold, (train_index, val_index) in enumerate(
    skf.split(X_train, y_train),
    start=1
):

    print(f"\nFold {fold}")
    print("Training size:", len(train_index))
    print("Validation size:", len(val_index))

    print("Validation category distribution:")
    print(y_train.iloc[val_index].value_counts())

    X_fold_train = X_train.iloc[train_index]
    X_fold_val = X_train.iloc[val_index]

    y_fold_train = y_train.iloc[train_index]
    y_fold_val = y_train.iloc[val_index]


Fold 1
Training size: 13440
Validation size: 3360
Validation category distribution:
categories
رياضة     560
اقتصاد    560
سياسة     560
دولي      560
مجتمع     560
ثقافة     560
Name: count, dtype: int64

Fold 2
Training size: 13440
Validation size: 3360
Validation category distribution:
categories
ثقافة     560
اقتصاد    560
سياسة     560
مجتمع     560
رياضة     560
دولي      560
Name: count, dtype: int64

Fold 3
Training size: 13440
Validation size: 3360
Validation category distribution:
categories
اقتصاد    560
سياسة     560
دولي      560
مجتمع     560
ثقافة     560
رياضة     560
Name: count, dtype: int64

Fold 4
Training size: 13440
Validation size: 3360
Validation category distribution:
categories
دولي      560
ثقافة     560
سياسة     560
مجتمع     560
رياضة     560
اقتصاد    560
Name: count, dtype: int64

Fold 5
Training size: 13440
Validation size: 3360
Validation category distribution:
categories
اقتصاد    560
رياضة     560
مجتمع     560
سياسة     560
دولي      560
ثقافة     